# Notebook for pseudobulk generating

- **N** равно числу клеток данной ткани в split'е — цель ~один псевдобалк на каждую train/test-клетку для каждого K.
- Якоря для bootstrap выбираются **случайно с возвращением** из пула (класс `Pseudobulk_Bootstrap`), поэтому при `N = |пул|` часть якорей может повториться (~63% уникальных в среднем).
- Имена колонок: `boot_K{K}_{tissue}_{i}`.

После генерации матрицы по всем тканям и K **склеиваются по колонкам** и перемешиваются (`sample(frac=1, axis=1)`).

---

## Ожидаемые размеры

| Split | Клеток | K | Итого псевдобалков |
|-------|--------|---|---------------------|
| Train | 1847 | × 5 | **9 235** |
| Test  | 463  | × 5 | **2 315** |

Распределение train-клеток по тканям: 293T-MS (431), A549-MS (500), HELA-MS (428), K562-MS (488).

---

## Особый случай K = 1

При `K = 1` класс возвращает **сырые single-cell профили** без агрегации, с **оригинальными баркодами** в качестве имён колонок. В данном ноутбуке K = 1 не используется.

---

## Связь с инференсом

Пайплайн KNN (log1p-CPM → HVG → PCA → NearestNeighbors → sum counts → TPM/CPM) согласован с `predict_knn_pseudobulk()` в `XGB/preprocessor.py`, чтобы обучение и предсказание работали на одном типе объектов.

In [1]:
import pandas as pd

test_b = pd.read_csv('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/barcodes_test_split.csv', index_col=0)
train_b = pd.read_csv('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/barcodes_train_split.csv', index_col=0)
all_b = pd.read_csv('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/barcodes_combined_filtered.csv', index_col=0)

In [2]:
len(set(test_b['barcode'].tolist()))

463

In [3]:
len(set(train_b['barcode'].tolist()))

1847

In [4]:
len(set(all_b['barcode'].tolist()))

2310

In [5]:
mirs_ = []
with open('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/targets.txt', 'r') as f:
        mirs_ = [line.strip() for line in f if line.strip()]

len(mirs_)

236

In [6]:
from sc_preprocessor import Pseudobulk_Sampler
path_mrna = '/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/mRNA_counts.csv'
path_mir = '/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/miRNA_counts.csv'
path_barcodes = '/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/barcodes_combined_filtered.csv' 
path_length = '/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/df_gene_mapping.parquet'
features = '/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/features.txt'

boot = Pseudobulk_Sampler(path_mrna, 
                            path_mir, 
                            path_barcodes, 
                            path_length, 
                            features,
                            mirs_)

Loading data...
Preprocessing miRNAs...
Ready. Cells: 2310, Tissues: 4


In [7]:
tissue_dict = {}
for tissue in boot.get_tissues():
    tissue_dict.setdefault(tissue, None)
    tissue_dict[tissue] = boot.get_tissue_barcodes(tissue).values

test_b = test_b['barcode'].tolist()
train_b = train_b['barcode'].tolist()

# GENERATE

In [8]:
import pandas as pd

train_features, train_targets = [], []
test_features_t,  test_targets_t  = {}, {}

for tissue in tissue_dict.keys():
    print(f"--- Processing Tissue: {tissue} ---")
    for k in [2, 3, 4, 5, 10]:
        
        # TRAIN
        current_size = len(set(tissue_dict[tissue]) & set(train_b))
        current_barcodes = list(set(tissue_dict[tissue]) & set(train_b))
        x_train, y_train = boot.generate(tissue, K=k, N=current_size, subset_barcodes=current_barcodes)
        train_features.append(x_train)
        train_targets.append(y_train)

        # TEST
        current_size = len(set(tissue_dict[tissue]) & set(test_b))
        current_barcodes = list(set(tissue_dict[tissue]) & set(test_b))
        x_test, y_test = boot.generate(tissue, K=k, N=current_size, subset_barcodes=current_barcodes)
        test_features_t.setdefault(k, [])
        test_targets_t.setdefault(k, [])
        test_features_t[k].append(x_test)
        test_targets_t[k].append(y_test)



# Склеиваем и перемешиваем (axis=1 так как у нас гены в строках)
# .sample(frac=1, axis=1) перемешивает порядок колонок
X_train_rna = pd.concat(train_features, axis=1).sample(frac=1, axis=1, random_state=42)
X_train_mir = pd.concat(train_targets, axis=1).sample(frac=1, axis=1, random_state=42)

X_test_rna = [pd.concat(test_features_t[k], axis=1) for k in test_features_t.keys()]
X_test_mir = [pd.concat(test_targets_t[k], axis=1) for k in test_targets_t.keys()]

print(f"\nFinal Shapes:")
print(f"Train RNA: {X_train_rna.shape} | Train miRNA: {X_train_mir.shape}")

--- Processing Tissue: 293T-MS ---
--- Processing Tissue: A549-MS ---
--- Processing Tissue: HELA-MS ---
--- Processing Tissue: K562-MS ---

Final Shapes:
Train RNA: (17392, 9235) | Train miRNA: (236, 9235)


In [17]:
X_test_rna[0].shape

(17392, 463)

# Pure single cell level

In [18]:
sc_train_features, sc_train_targets = [], []
sc_test_features,  sc_test_targets = [], []

for tissue in tissue_dict.keys():
    print(f"--- Processing Tissue: {tissue} ---")

    # TRAIN
    current_size = len(set(tissue_dict[tissue]) & set(train_b))
    current_barcodes = list(set(tissue_dict[tissue]) & set(train_b))
    x_train, y_train = boot.generate(tissue, K=1, N=current_size, subset_barcodes=current_barcodes)
    sc_train_features.append(x_train)
    sc_train_targets.append(y_train)

    # TEST
    current_size = len(set(tissue_dict[tissue]) & set(test_b))
    current_barcodes = list(set(tissue_dict[tissue]) & set(test_b))
    x_test, y_test = boot.generate(tissue, K=1, N=current_size, subset_barcodes=current_barcodes)
    sc_test_features.append(x_test)
    sc_test_targets.append(y_test)

X_train_rna_sc = pd.concat(sc_train_features, axis=1)
X_train_mir_sc = pd.concat(sc_train_targets, axis=1)

X_test_rna_sc = pd.concat(sc_test_features, axis=1)
X_test_mir_sc = pd.concat(sc_test_targets, axis=1)

print(f"\nFinal Shapes:")
print(f"Train RNA: {X_train_rna_sc.shape} | Train miRNA: {X_train_mir_sc.shape}")
print(f"Test RNA: {X_test_rna_sc.shape} | Test miRNA: {X_test_mir_sc.shape}")

--- Processing Tissue: 293T-MS ---
--- Processing Tissue: A549-MS ---
--- Processing Tissue: HELA-MS ---
--- Processing Tissue: K562-MS ---

Final Shapes:
Train RNA: (17392, 1847) | Train miRNA: (236, 1847)
Test RNA: (17392, 463) | Test miRNA: (236, 463)


In [20]:
X_train_rna_sc.sum(axis=0)

AACCAAAGGATG_1    1000000.0
AACCAAATAGAG_2    1000000.0
AACCAACCTGTT_2    1000000.0
AACCAAGAATGA_1    1000000.0
AACCTCATGTGG_1    1000000.0
                    ...    
TTCTGACAGATT_2    1000000.0
TTCTGAGTATGC_1    1000000.0
TTCTGAGTCGTG_2    1000000.0
TTCTGATACATC_1    1000000.0
TTCTGATTAGCC_1    1000000.0
Length: 1847, dtype: float64

In [21]:
X_train_rna.sum(axis=0)

boot_K3_HELA-MS_273     1000000.0
boot_K4_HELA-MS_135     1000000.0
boot_K3_293T-MS_161     1000000.0
boot_K10_293T-MS_409    1000000.0
boot_K5_K562-MS_343     1000000.0
                          ...    
boot_K4_HELA-MS_223     1000000.0
boot_K3_HELA-MS_108     1000000.0
boot_K3_HELA-MS_307     1000000.0
boot_K3_293T-MS_429     1000000.0
boot_K2_K562-MS_475     1000000.0
Length: 9235, dtype: float64

In [22]:
X_test_rna_sc.sum(axis=0)

AACCAACCTGTT_1    1000000.0
AACCTCTGCGTA_1    1000000.0
AACGGTATAGAG_1    1000000.0
ACCAGATTGTCG_1    1000000.0
ACTAAGCGTTAG_2    1000000.0
                    ...    
TCGCGGTCAGCT_2    1000000.0
TCGGCAAATCTG_1    1000000.0
TCGGCAAGGTGC_1    1000000.0
TCGGCATCAGCT_2    1000000.0
TTACTTTCGCTA_1    1000000.0
Length: 463, dtype: float64

In [23]:
X_test_rna[0].sum(axis=0)

boot_K2_293T-MS_0      1000000.0
boot_K2_293T-MS_1      1000000.0
boot_K2_293T-MS_2      1000000.0
boot_K2_293T-MS_3      1000000.0
boot_K2_293T-MS_4      1000000.0
                         ...    
boot_K2_K562-MS_118    1000000.0
boot_K2_K562-MS_119    1000000.0
boot_K2_K562-MS_120    1000000.0
boot_K2_K562-MS_121    1000000.0
boot_K2_K562-MS_122    1000000.0
Length: 463, dtype: float64

In [32]:
assert all(X_train_rna_sc.columns == X_train_mir_sc.columns)
assert all(X_test_rna_sc.columns == X_test_mir_sc.columns)

assert all(X_train_rna.columns == X_train_mir.columns)

for i in range(len(X_test_rna)):
    assert all(X_test_rna[i].columns == X_test_mir[i].columns)

### SAVE

In [26]:
X_train_rna.to_parquet('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TRAIN/X_SS_TRAIN_PB.parquet')
X_train_mir.to_parquet('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TRAIN/Y_SS_TRAIN_PB.parquet')

X_train_rna_sc.to_parquet('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TRAIN/X_SS_TRAIN_SC.parquet')
X_train_mir_sc.to_parquet('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TRAIN/Y_SS_TRAIN_SC.parquet')

In [33]:
for i in range(len(X_test_rna)):
    X_test_rna[i].to_parquet(f'/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TEST/X_SS_TEST_PB_K{i}.parquet')
    X_test_mir[i].to_parquet(f'/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TEST/Y_SS_TEST_PB_K{i}.parquet')

X_test_rna_sc.to_parquet('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TEST/X_SS_TEST_SC.parquet')
X_test_mir_sc.to_parquet('/mnt/jack-5/amismailov/miRNA_study/PREPARE_SC_DATA/TEST/Y_SS_TEST_SC.parquet')